# Web Search

*Notebook 08*

Give your agent access to current information from the web.

No extra search API key required.


---

## 🔧 Setup

The main path starts four paid agent runs.

Three can use web search, which may add search tool-call cost.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, MessageOutputItem, Runner, ToolCallItem, WebSearchTool
from IPython.display import display, Markdown

MODEL = "gpt-5-mini"


def used_web_search(result) -> bool:
    """True if the run actually invoked the hosted web search tool."""
    return any(
        isinstance(item, ToolCallItem)
        and getattr(item.raw_item, "type", None) == "web_search_call"
        for item in result.new_items
    )


def get_url_citations(result):
    """Collect URL citations attached to assistant output messages."""
    citations = []
    for item in result.new_items:
        if not isinstance(item, MessageOutputItem):
            continue
        for content in item.raw_item.content:
            for annotation in getattr(content, "annotations", []):
                if getattr(annotation, "type", None) == "url_citation":
                    citations.append(annotation)
    return citations


def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


# --------------------------------------------------------------
# Verify setup
# --------------------------------------------------------------
print("✅ Ready!")

⚠️ **Security note:** Web content may contain prompt injection.

Treat retrieved text as data, not instructions (Lesson 21).

---

## 🎯 The Problem

Models have knowledge cutoffs.

Ask about last week's news or today's weather.

Web search lets the agent check current sources.

---

## 🌐 Part 1: Enabling Web Search

Add `WebSearchTool()` to the agent's `tools` list.

#### Without Web Search

In [ ]:
comparison_instructions = (
    "Answer questions about current information as accurately as you can.\n"
    "Keep the answer to two short bullets."
)

version_question = (
    "What is the latest stable version of Python, "
    "and when was it released?"
)

no_search_agent = Agent(
    name="NoSearchAgent",
    instructions=comparison_instructions,
    model=MODEL
)

result = await Runner.run(no_search_agent, input=version_question)

print("Without web search:")
print(f"Web search called: {'yes' if used_web_search(result) else 'no'}")
display(Markdown(result.final_output))

#### With Web Search

In [ ]:
web_search_agent = Agent(
    name="WebSearchAgent",
    instructions=comparison_instructions,
    model=MODEL,
    tools=[WebSearchTool()]
)

result = await Runner.run(web_search_agent, input=version_question)

print("With web search:")
print(f"Web search called: {'yes' if used_web_search(result) else 'no'}")
display(Markdown(result.final_output))

### 💡 Key Takeaway

`WebSearchTool()` lets the agent ground answers in retrieved web content.

The agent still decides whether to search.

---

## 📎 Part 2: Source Citations

When web search is used, cited URLs arrive as response annotations.

Inspect those annotations instead of inferring citations from prose.

OpenAI requires inline citations to be visible and clickable.

In [ ]:
citation_instructions = (
    "Search the web for the answer.\n"
    "Return three short bullets."
)

citation_agent = Agent(
    name="CitationAgent",
    instructions=citation_instructions,
    model=MODEL,
    tools=[WebSearchTool()]
)

features_question = (
    "What are the headline new features in the latest stable Python release?"
)

result = await Runner.run(citation_agent, input=features_question)
citations = get_url_citations(result)

print(f"Web search called: {'yes' if used_web_search(result) else 'no'}")
display(Markdown(result.final_output))

print(f"Attached URL citations: {len(citations)}")
if citations:
    for citation in citations:
        title = citation.title or citation.url
        display(Markdown(f"- [{title}]({citation.url})"))
else:
    print("⚠️  No URL citations were attached to this answer")

### 💡 Key Takeaway

An answer can name sources in prose without carrying citation annotations.

---

## 📍 Part 3: Location-Targeted Search

`user_location` biases results toward an approximate geography.

Useful for local news, weather, and business searches.

These settings are checked when the request reaches the API, not locally.

In [ ]:
# Change these to your location
# "approximate" is the only supported type for user_location
CITY = "Denver"
REGION = "Colorado"
COUNTRY = "US"

local_instructions = (
    "Search the web before answering local information questions."
)

local_search_agent = Agent(
    name="LocalSearchAgent",
    instructions=local_instructions,
    model=MODEL,
    tools=[
        WebSearchTool(
            user_location={
                "type": "approximate",
                "city": CITY,
                "region": REGION,
                "country": COUNTRY,
            }
        )
    ]
)

weather_question = "What is the weather like today?"
result = await Runner.run(local_search_agent, input=weather_question)
citations = get_url_citations(result)

print(f"Web search called: {'yes' if used_web_search(result) else 'no'}")
display(Markdown(truncate_response(result.final_output)))

if citations:
    print("Sources:")
    for citation in citations:
        title = citation.title or citation.url
        display(Markdown(f"- [{title}]({citation.url})"))

### 💡 Key Takeaway

Location bias guides ranking without restricting results to one geography.

---

## 💪 Practice Exercises

### Exercise 1: Tech News Agent

*Covers: `WebSearchTool`, summarizing recent web results*

Create a three-bullet news summary for a topic you choose.

Costs one paid agent run and may add search tool-call cost.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Tech News Agent
# --------------------------------------------------------------
# Objective: Build an agent that fetches and summarizes recent news.

topic = "Python programming language"  # Change to any topic you like

# TODO 1: Create an Agent with WebSearchTool.
#          Instruct it to search before answering in three bullets.

# TODO 2: Run the agent with a news query about your topic.

# TODO 3: Display the truncated response and confirm:
#            - used_web_search(result) is True
#            - the answer is a short three-bullet summary, not an essay
#          Render any get_url_citations(result) sources separately.

# --- Write your code below this line ---

### Exercise 2: Fact Checker

*Covers: `WebSearchTool`, claim verification with citations*

Verify a claim and support the verdict with web sources.

Costs one paid agent run and may add search tool-call cost.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Fact Checker
# --------------------------------------------------------------
# Objective: Build an agent that verifies a claim with web sources.

claim = "Python 3.0 was released in 2008"

# TODO 1: Create an Agent with WebSearchTool.
#          Instruct it to search and cite sources before answering.
#          Return a verdict of supported, unsupported, or mixed.

# TODO 2: Run the agent and ask it to verify the claim.

# TODO 3: Display the response and confirm:
#            - used_web_search(result) is True
#            - the verdict is supported, unsupported, or mixed
#            - get_url_citations(result) returns at least one source

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`WebSearchTool` adds current context and a trust boundary:**

- Add `WebSearchTool()` to the agent's `tools`, with no extra search API key

- The agent decides whether to search, guided by the question and instructions

- Treat retrieved text as untrusted data, not instructions
<br>
<br>

**Treat citations as structured evidence:**

- Search-backed answers carry cited URLs in response annotations

- Inspect annotations instead of inferring citations from prose

- Render cited sources visibly and clickably
<br>
<br>

**Bias search toward a location:**

- Pass an approximate city, region, and country through `user_location`

- Broad queries may still return global results despite the location bias

- Invalid location settings fail when the run reaches the API

---

## 📍 Next Step

**Notebook 09: File Search**  

Search your own documents with OpenAI's built-in vector search.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-08-web-search)

---